In [0]:
import os, re, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from databricks.sdk import WorkspaceClient
from lh import common, jobs

# t28_jobs: the deployed daily jobs (dev, test, prod) match config/jobs.yml exactly.
# Run by the Developer from the dev Git folder; read-only (changes nothing, starts no run).
# FAIL = the job differs from the config; WARN = could not be read with this login, or no recent dev run.
root = os.path.dirname(os.getcwd())
jcfg = jobs.load_jobs(root)
api = jobs.Api(WorkspaceClient())
UUID = re.compile(r"^[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}$")
results = []


def check(level, name, ok, detail=""):
    status = "PASS" if ok else level
    results.append(status)
    print(f"{status:4} {name}", detail)


def shape(tasks):
    """The parts of each task that config/jobs.yml decides."""
    out = {}
    for t in tasks:
        nb = t.get("notebook_task", {})
        out[t["task_key"]] = (nb.get("notebook_path"), nb.get("source", "WORKSPACE"),
                              tuple(sorted(nb.get("base_parameters", {}).items())),
                              t.get("timeout_seconds"), t.get("max_retries", 0),
                              tuple(sorted(d["task_key"] for d in t.get("depends_on", []))),
                              t.get("run_if", "ALL_SUCCESS"))
    return out


code = common.load_config("dev")["code"]
jobs.check(jobs.tasks_for(jcfg, code), root)
check("FAIL", "config/jobs.yml: tasks, dependencies and notebooks valid", True)

for env in ("dev", "test", "prod"):
    name = jcfg["name"].format(code=code, env=env)
    folder = jcfg["folder"][env].format(repo=jcfg["repo"][code])
    print(f"\n--- {name}")
    found = api.call("GET", "/api/2.2/jobs/list", query={"name": name}).get("jobs", [])
    check("FAIL", f"{name}: exactly one job", len(found) == 1, f"(found {len(found)})")
    if len(found) != 1:
        continue
    job = api.get_job(found[0]["job_id"])
    s = job["settings"]

    # where the notebooks come from: this environment's own Git folder
    paths = [t.get("notebook_task", {}).get("notebook_path", "") for t in s.get("tasks", [])]
    homes = {p.split(f"/{folder}/")[0] for p in paths if f"/{folder}/" in p}
    check("FAIL", f"{name}: every task runs from the {folder} Git folder", len(homes) == 1 and len(paths) == len(s.get("tasks", [])) and all(f"/{folder}/" in p for p in paths))
    home = homes.pop() if len(homes) == 1 else "/Users/unknown"

    # run as: dev = the deployer (a user), test/prod = the environment's job account
    run_as = s.get("run_as", {}).get("service_principal_name") or job.get("run_as_user_name", "")
    if env == "dev":
        check("FAIL", f"{name}: runs as the deployer, not a job account", not UUID.match(run_as or "x") and "service_principal_name" not in s.get("run_as", {}))
        check("FAIL", f"{name}: no schedule (on demand)", "schedule" not in s and "trigger" not in s)
    else:
        check("FAIL", f"{name}: runs as a job account", bool(UUID.match(run_as or "")))
        sch = s.get("schedule", {})
        want_pause = "PAUSED" if jcfg["paused"] else "UNPAUSED"
        check("FAIL", f"{name}: schedule {jcfg['schedule'][code]} {jcfg['timezone']} {want_pause}",
              (sch.get("quartz_cron_expression"), sch.get("timezone_id"), sch.get("pause_status"))
              == (jcfg["schedule"][code], jcfg["timezone"], want_pause),
              "" if sch else "(no schedule)")

    # tasks, dependencies, run_if, timeouts, retries, parameters = exactly what build() makes from the config
    want = jobs.build(jcfg, code, env, home, run_as if env != "dev" else None)
    have_t, want_t = shape(s.get("tasks", [])), shape(want["tasks"])
    diff = sorted(k for k in set(have_t) | set(want_t) if have_t.get(k) != want_t.get(k))
    check("FAIL", f"{name}: tasks exactly as config ({' -> '.join(t['task_key'] for t in want['tasks'])})", not diff, diff or "")
    check("FAIL", f"{name}: job timeout, one run at a time, queue, tags",
          (s.get("timeout_seconds"), s.get("max_concurrent_runs"), s.get("queue", {}).get("enabled"), s.get("tags"))
          == (want["timeout_seconds"], 1, True, want["tags"]))

    # permissions: owner + groups exactly as config (the inherited admins entry is ignored)
    try:
        acl = api.get_acl(found[0]["job_id"]).get("access_control_list", [])
    except Exception as e:
        check("WARN", f"{name}: permissions readable with this login", False, f"({type(e).__name__}; check as Management/admin)")
        acl = None
    if acl is not None:
        direct = {(a.get("group_name") or ("job account" if a.get("service_principal_name") else "a user"), p["permission_level"])
                  for a in acl for p in a.get("all_permissions", []) if not p.get("inherited")}
        groups = {(g, l) for g, l in direct if l != "IS_OWNER"}
        want_g = set(jcfg["permissions"][env].items())
        check("FAIL", f"{name}: group permissions exactly as config", groups == want_g,
              "" if groups == want_g else f"missing {sorted(want_g - groups)} extra {sorted(groups - want_g)}")
        owners = [a for a in acl for p in a.get("all_permissions", []) if p["permission_level"] == "IS_OWNER"]
        if env == "dev":
            check("FAIL", f"{name}: owner is the deployer", len(owners) == 1 and "user_name" in owners[0])
        else:
            sp = jcfg["run_as"][env].format(code=code)
            check("FAIL", f"{name}: owner and run-as are {sp}", len(owners) == 1
                  and owners[0].get("service_principal_name") == run_as and owners[0].get("display_name") == sp)
        check("FAIL", f"{name}: no access for all workspace users", not any(a.get("group_name") == "users" for a in acl))

    # runs: dev proven on demand; test/prod not started before promotion (5.10)
    runs = api.call("GET", "/api/2.2/jobs/runs/list", query={"job_id": found[0]["job_id"], "limit": 5, "completed_only": "true"}).get("runs", [])
    states = [r.get("state", {}).get("result_state") for r in runs]
    if env == "dev":
        check("WARN", f"{name}: last finished run succeeded", bool(states) and states[0] in ("SUCCESS", "SUCCESS_WITH_FAILURES"), states[:1] or "(no run)")
    else:
        check("WARN", f"{name}: no runs before promotion (5.10)", not runs, f"({len(runs)} runs)" if runs else "")

print(f"\n{results.count('PASS')}/{len(results)} PASS, {results.count('WARN')} WARN, {results.count('FAIL')} FAIL")
assert "FAIL" not in results, "t28_jobs failed"